# Proyecto Mundial: Exploracion inicial de datos medicos

Este notebook documenta la primera parte del proyecto: cargar el conjunto de datos, entender su estructura, revisar las features, calcular rangos de valores y visualizar frecuencias. La variable `condition` se trata como variable objetivo, por lo que los analisis de features la excluyen por defecto.


## 1. Preparacion del entorno

Ahora preparamos las librerias y las rutas del proyecto. Esto permite que el notebook funcione aunque se ejecute desde la carpeta del notebook o desde la raiz del repositorio.


In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "data" / "foot.csv").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR / "mundial_project").exists():
    PROJECT_ROOT = NOTEBOOK_DIR / "mundial_project"
else:
    PROJECT_ROOT = NOTEBOOK_DIR.parent

DATA_PATH = PROJECT_ROOT / "data" / "foot.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "graficas_frecuencia"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH, OUTPUT_DIR


## 2. Carga de datos

Ahora estamos en la parte de analisis inicial: cargamos el CSV para comprobar dimensiones, columnas y primeras filas. Esto confirma que el dataset se lee correctamente antes de hacer cualquier transformacion o grafica.


In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()


## 3. Estructura del dataset

Ahora revisamos tipos de datos, valores no nulos y estadisticos generales. Esta exploracion sirve para detectar columnas numericas, posibles valores faltantes y escalas muy diferentes entre features.


In [ ]:
df.info()
df.describe(include="all")


## 4. Separacion de features y variable objetivo

Ahora definimos `condition` como la variable que queremos predecir mas adelante. Las demas columnas se consideran features, por eso las analizaremos sin mezclar la variable objetivo con las entradas del modelo.


In [ ]:
target = "condition"
features = [column for column in df.columns if column != target]

print(f"Variable objetivo: {target}")
print(f"Numero de features: {len(features)}")
features


## 5. Rangos de valores de las features

Ahora revisamos los rangos de cada feature para entender sus escalas, minimos, maximos y posibles anomalias. Esto es importante porque algunos modelos son sensibles a la escala y porque un rango inesperado puede indicar datos mal codificados.


In [ ]:
feature_ranges = []

for feature in features:
    series = df[feature]
    if pd.api.types.is_numeric_dtype(series):
        minimum = series.min()
        maximum = series.max()
        value_range = maximum - minimum
    else:
        minimum = None
        maximum = None
        value_range = None

    feature_ranges.append(
        {
            "feature": feature,
            "tipo_dato": str(series.dtype),
            "minimo": minimum,
            "maximo": maximum,
            "rango": value_range,
            "valores_unicos": series.nunique(dropna=True),
            "valores_nulos": series.isna().sum(),
        }
    )

rangos_features = pd.DataFrame(feature_ranges)
rangos_features


## 6. Exportacion de rangos como imagen

Ahora exportamos la tabla de rangos como una imagen PNG. Esto deja una evidencia visual directa en `outputs/` para incluirla facilmente en el reporte o revisar las escalas sin volver a ejecutar el notebook.


In [ ]:
display_ranges = rangos_features.copy()
for column in ("minimo", "maximo", "rango"):
    display_ranges[column] = display_ranges[column].map(
        lambda value: f"{value:g}" if pd.notna(value) else ""
    )

image_path = PROJECT_ROOT / "outputs" / "rangos_features.png"
fig_height = max(3.5, len(display_ranges) * 0.35 + 1.2)
fig, ax = plt.subplots(figsize=(12, fig_height))
ax.axis("off")

table = ax.table(
    cellText=display_ranges.values,
    colLabels=display_ranges.columns,
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.35)

for (row, _), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold", color="white")
        cell.set_facecolor("#4C78A8")
    elif row % 2 == 0:
        cell.set_facecolor("#F2F4F7")

fig.suptitle("Rangos de valores por feature", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(image_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print(f"Imagen guardada en: {image_path.resolve()}")


## 7. Estadisticas de resumen

Ahora calculamos la media (mu) y la varianza (sigma^2) de cada feature numerica. La media nos da una referencia del valor central y la varianza indica que tan dispersos estan los datos; juntas ayudan a entender la distribucion y a anticipar si sera necesario escalar variables antes de entrenar modelos.


In [ ]:
summary_stats = []

for feature in features:
    series = df[feature]
    if pd.api.types.is_numeric_dtype(series):
        mean = series.mean()
        variance = series.var()
    else:
        mean = None
        variance = None

    summary_stats.append(
        {
            "feature": feature,
            "tipo_dato": str(series.dtype),
            "media_mu": mean,
            "varianza_sigma2": variance,
            "valores_unicos": series.nunique(dropna=True),
            "valores_nulos": series.isna().sum(),
        }
    )

estadisticas_resumen = pd.DataFrame(summary_stats)
estadisticas_resumen


## 8. Exportacion de estadisticas como imagen

Ahora guardamos la tabla de media y varianza como imagen. Esto facilita incluir estas estadisticas de resumen en el reporte sin depender de capturas manuales del notebook.


In [ ]:
display_stats = estadisticas_resumen.copy()
for column in ("media_mu", "varianza_sigma2"):
    display_stats[column] = display_stats[column].map(
        lambda value: f"{value:.4f}" if pd.notna(value) else ""
    )

stats_image_path = PROJECT_ROOT / "outputs" / "estadisticas_resumen.png"
fig_height = max(3.5, len(display_stats) * 0.35 + 1.2)
fig, ax = plt.subplots(figsize=(12, fig_height))
ax.axis("off")

table = ax.table(
    cellText=display_stats.values,
    colLabels=display_stats.columns,
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.35)

for (row, _), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold", color="white")
        cell.set_facecolor("#4C78A8")
    elif row % 2 == 0:
        cell.set_facecolor("#F2F4F7")

fig.suptitle("Estadisticas de resumen por feature", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(stats_image_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print(f"Imagen guardada en: {stats_image_path.resolve()}")


## 9. Deteccion de outliers con IQR

Ahora detectamos valores atipicos con el metodo IQR. Antes excluimos features booleanas o discretas de rango pequeno, como `feature_3`, porque en esas columnas los valores distintos representan categorias codificadas y no desviaciones reales. El IQR se aplica solo a features numericas con suficiente variabilidad.


In [ ]:
OUTLIER_DIR = PROJECT_ROOT / "outputs" / "outliers"
OUTLIER_DIR.mkdir(parents=True, exist_ok=True)
MIN_UNIQUE_VALUES = 6
MIN_VALUE_RANGE = 3


def is_outlier_candidate(series):
    if not pd.api.types.is_numeric_dtype(series):
        return False
    clean_series = series.dropna()
    if clean_series.nunique() < MIN_UNIQUE_VALUES:
        return False
    return (clean_series.max() - clean_series.min()) > MIN_VALUE_RANGE


outlier_features = [
    feature for feature in features if is_outlier_candidate(df[feature])
]
excluded_outlier_features = [
    feature for feature in features if feature not in outlier_features
]

print(f"Features usadas para outliers: {outlier_features}")
print(f"Features excluidas por ser booleanas o de rango pequeno: {excluded_outlier_features}")


def save_dataframe_image(table_df, output_path, title, figsize=(14, 4)):
    display_df = table_df.copy()
    numeric_columns = display_df.select_dtypes(include="number").columns
    for column in numeric_columns:
        display_df[column] = display_df[column].map(lambda value: f"{value:.4f}")

    fig_height = max(figsize[1], len(display_df) * 0.35 + 1.2)
    fig, ax = plt.subplots(figsize=(figsize[0], fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.35)

    for (row, _), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#4C78A8")
        elif row % 2 == 0:
            cell.set_facecolor("#F2F4F7")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


iqr_rows = []
iqr_outlier_indices = {}
iqr_multiplier = 1.5

for feature in outlier_features:
    series = df[feature]
    if not pd.api.types.is_numeric_dtype(series):
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - iqr_multiplier * iqr
    upper_limit = q3 + iqr_multiplier * iqr
    mask = (series < lower_limit) | (series > upper_limit)
    indices = set(df.index[mask].tolist())
    iqr_outlier_indices[feature] = indices

    iqr_rows.append(
        {
            "feature": feature,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "limite_inferior": lower_limit,
            "limite_superior": upper_limit,
            "outliers_iqr": len(indices),
            "porcentaje_iqr": len(indices) / len(df) * 100,
        }
    )

outliers_iqr = pd.DataFrame(iqr_rows)
outliers_iqr.to_csv(OUTLIER_DIR / "outliers_iqr.csv", index=False)
save_dataframe_image(outliers_iqr, OUTLIER_DIR / "outliers_iqr.png", "Outliers por metodo IQR")
outliers_iqr


## 10. Deteccion con residuos estandarizados y QQ-plots

Ahora usamos residuos estandarizados solo sobre las features continuas o de rango amplio seleccionadas. Medimos cuantas desviaciones estandar se aleja cada valor de la media y marcamos como outliers los puntos con `|z| > 3`. Tambien generamos QQ-plots para ver si las colas se separan del comportamiento normal esperado.


In [ ]:
from statistics import NormalDist


def normal_quantiles(size):
    normal = NormalDist()
    return [normal.inv_cdf((rank - 0.5) / size) for rank in range(1, size + 1)]


z_rows = []
z_outlier_indices = {}
z_threshold = 3.0

for feature in outlier_features:
    series = df[feature]
    if not pd.api.types.is_numeric_dtype(series):
        continue

    std = series.std()
    if std == 0 or pd.isna(std):
        z_scores = pd.Series(0.0, index=series.index)
    else:
        z_scores = (series - series.mean()) / std

    mask = z_scores.abs() > z_threshold
    indices = set(df.index[mask].tolist())
    z_outlier_indices[feature] = indices

    sorted_values = series.dropna().sort_values().reset_index(drop=True)
    theoretical = pd.Series(normal_quantiles(len(sorted_values)))
    qq_correlation = sorted_values.corr(theoretical) if len(sorted_values) > 1 else None

    z_rows.append(
        {
            "feature": feature,
            "media": series.mean(),
            "desviacion_std": std,
            "umbral_z": z_threshold,
            "outliers_zscore": len(indices),
            "porcentaje_zscore": len(indices) / len(df) * 100,
            "qq_correlacion": qq_correlation,
        }
    )

outliers_residuos_qq = pd.DataFrame(z_rows)
outliers_residuos_qq.to_csv(OUTLIER_DIR / "outliers_residuos_qq.csv", index=False)
save_dataframe_image(
    outliers_residuos_qq,
    OUTLIER_DIR / "outliers_residuos_qq.png",
    "Outliers por residuos estandarizados",
)

cols = 3
rows = math.ceil(len(outlier_features) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5.0, rows * 4.0))
flat_axes = axes.ravel() if hasattr(axes, "ravel") else [axes]

for ax, feature in zip(flat_axes, outlier_features):
    values = df[feature].dropna().sort_values().reset_index(drop=True)
    theoretical = pd.Series(normal_quantiles(len(values)))
    ax.scatter(theoretical, values, s=12, alpha=0.7, color="#4C78A8")
    ax.set_title(feature)
    ax.set_xlabel("Cuantiles teoricos normales")
    ax.set_ylabel("Valores observados")
    ax.grid(alpha=0.25)

for ax in flat_axes[len(outlier_features):]:
    ax.set_visible(False)

fig.suptitle("QQ-plots por feature", fontsize=16, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.98))
fig.savefig(OUTLIER_DIR / "qq_plots.png", dpi=180, bbox_inches="tight")
plt.show()

outliers_residuos_qq


## 11. Comparacion entre metodos de outliers

Ahora comparamos si IQR y residuos estandarizados detectan los mismos registros. Esta comparacion es importante porque los metodos no responden igual: IQR suele ser mas robusto ante colas y variables discretas, mientras que z-score asume mejor comportamiento cuando la feature se parece mas a una distribucion normal.


In [ ]:
comparison_rows = []

for feature in sorted(set(iqr_outlier_indices) | set(z_outlier_indices)):
    iqr_set = iqr_outlier_indices.get(feature, set())
    z_set = z_outlier_indices.get(feature, set())
    both = iqr_set & z_set
    union = iqr_set | z_set
    agreement = len(both) / len(union) * 100 if union else 100.0

    comparison_rows.append(
        {
            "feature": feature,
            "outliers_iqr": len(iqr_set),
            "outliers_zscore": len(z_set),
            "coinciden": len(both),
            "solo_iqr": len(iqr_set - z_set),
            "solo_zscore": len(z_set - iqr_set),
            "acuerdo_porcentaje": agreement,
        }
    )

comparacion_outliers = pd.DataFrame(comparison_rows)
comparacion_outliers.to_csv(OUTLIER_DIR / "comparacion_outliers.csv", index=False)
save_dataframe_image(
    comparacion_outliers,
    OUTLIER_DIR / "comparacion_outliers.png",
    "Comparacion de outliers: IQR vs z-score",
    figsize=(13, 4),
)
comparacion_outliers


## 12. Validacion de outliers con SciPy

Ahora repetimos el experimento de residuos estandarizados y QQ-plots usando `scipy`, manteniendo el mismo filtro de features candidatas. SciPy es una libreria estadistica mas especializada, por lo que `stats.zscore` y `stats.probplot` son una referencia mas estandar para validar que nuestro calculo manual no cambia los resultados.


In [ ]:
from scipy import stats

scipy_rows = []
scipy_outlier_indices = {}

for feature in outlier_features:
    series = df[feature]
    if not pd.api.types.is_numeric_dtype(series):
        continue

    z_scores = pd.Series(
        stats.zscore(series, nan_policy="omit", ddof=1),
        index=series.index,
    ).fillna(0.0)

    mask = z_scores.abs() > z_threshold
    indices = set(df.index[mask].tolist())
    scipy_outlier_indices[feature] = indices

    (_, _), (_, _, qq_r) = stats.probplot(series.dropna(), dist="norm")
    scipy_rows.append(
        {
            "feature": feature,
            "media": series.mean(),
            "desviacion_std": series.std(),
            "umbral_z": z_threshold,
            "outliers_scipy": len(indices),
            "porcentaje_scipy": len(indices) / len(df) * 100,
            "qq_r_scipy": qq_r,
        }
    )

outliers_scipy = pd.DataFrame(scipy_rows)
outliers_scipy.to_csv(OUTLIER_DIR / "outliers_scipy.csv", index=False)
save_dataframe_image(outliers_scipy, OUTLIER_DIR / "outliers_scipy.png", "Outliers con SciPy")

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5.0, rows * 4.0))
flat_axes = axes.ravel() if hasattr(axes, "ravel") else [axes]

for ax, feature in zip(flat_axes, outlier_features):
    stats.probplot(df[feature].dropna(), dist="norm", plot=ax)
    ax.set_title(feature)
    ax.grid(alpha=0.25)

for ax in flat_axes[len(outlier_features):]:
    ax.set_visible(False)

fig.suptitle("QQ-plots con SciPy por feature", fontsize=16, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.98))
fig.savefig(OUTLIER_DIR / "qq_plots_scipy.png", dpi=180, bbox_inches="tight")
plt.show()

outliers_scipy


## 13. Comparacion SciPy vs calculo manual

Ahora comparamos los outliers detectados por SciPy contra los residuos estandarizados calculados manualmente. Si el acuerdo es 100%, podemos mantener la interpretacion anterior con mas confianza; si hay diferencias, se revisan como una decision metodologica.


In [ ]:
scipy_comparison_rows = []

for feature in sorted(set(z_outlier_indices) | set(scipy_outlier_indices)):
    manual_set = z_outlier_indices.get(feature, set())
    scipy_set = scipy_outlier_indices.get(feature, set())
    both = manual_set & scipy_set
    union = manual_set | scipy_set
    agreement = len(both) / len(union) * 100 if union else 100.0

    scipy_comparison_rows.append(
        {
            "feature": feature,
            "outliers_manual": len(manual_set),
            "outliers_scipy": len(scipy_set),
            "coinciden": len(both),
            "solo_manual": len(manual_set - scipy_set),
            "solo_scipy": len(scipy_set - manual_set),
            "acuerdo_porcentaje": agreement,
        }
    )

comparacion_scipy_manual = pd.DataFrame(scipy_comparison_rows)
comparacion_scipy_manual.to_csv(OUTLIER_DIR / "comparacion_scipy_manual.csv", index=False)
save_dataframe_image(
    comparacion_scipy_manual,
    OUTLIER_DIR / "comparacion_scipy_manual.png",
    "Comparacion SciPy vs calculo manual",
)
comparacion_scipy_manual


## 14. Matriz de correlacion

Ahora calculamos una matriz de correlacion entre las columnas numericas. Esto nos ayuda a ver que features se mueven juntas, cuales podrian aportar informacion redundante y cuales tienen una relacion lineal directa o inversa con `condition`. Usamos Pearson como primera lectura porque resume relaciones lineales entre variables numericas.


In [ ]:
CORRELATION_DIR = PROJECT_ROOT / "outputs" / "correlacion"
CORRELATION_DIR.mkdir(parents=True, exist_ok=True)

correlation_matrix = df.select_dtypes(include="number").corr(method="pearson")
correlation_matrix.to_csv(CORRELATION_DIR / "matriz_correlacion.csv")
correlation_matrix


## 15. Heatmap de correlacion

Ahora convertimos la matriz en un mapa de calor. Los colores cercanos a rojo indican correlacion positiva fuerte, los cercanos a azul indican correlacion negativa fuerte, y los tonos claros indican relaciones lineales debiles o casi nulas.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
image = ax.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Correlacion")

ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_yticks(range(len(correlation_matrix.index)))
ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha="right")
ax.set_yticklabels(correlation_matrix.index)

for row in range(len(correlation_matrix.index)):
    for col in range(len(correlation_matrix.columns)):
        value = correlation_matrix.iloc[row, col]
        color = "white" if abs(value) >= 0.55 else "black"
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", color=color, fontsize=8)

ax.set_title("Matriz de correlacion (Pearson)", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(CORRELATION_DIR / "matriz_correlacion.png", dpi=180, bbox_inches="tight")
plt.show()


## 16. Estandarizacion de las 13 caracteristicas

Ahora estandarizamos las 13 features restando la media `mu` y dividiendo entre la desviacion estandar `sigma`: `z = (x - mu) / sigma`. Esto deja cada feature con media aproximada 0 y varianza 1, lo que permite comparar todas las variables en la misma escala antes de analizar relaciones multivariadas. Aunque la correlacion de Pearson es invariante a cambios lineales de escala, esta transformacion deja el experimento alineado con los pasos matematicos del proyecto y prepara los datos para tecnicas posteriores.


In [ ]:
STANDARDIZATION_DIR = PROJECT_ROOT / "outputs" / "estandarizacion_correlacion"
STANDARDIZATION_DIR.mkdir(parents=True, exist_ok=True)

feature_df = df[features]
features_estandarizadas = (feature_df - feature_df.mean()) / feature_df.std()
features_estandarizadas.to_csv(
    STANDARDIZATION_DIR / "features_estandarizadas.csv",
    index=False,
)
features_estandarizadas.head()


## 17. Comprobacion de la estandarizacion

Ahora verificamos que la transformacion hizo lo esperado: cada feature estandarizada debe tener media muy cercana a 0, varianza cercana a 1 y desviacion estandar cercana a 1. Esta tabla funciona como control de calidad antes de calcular la matriz de correlacion.


In [ ]:
comprobacion_estandarizacion = pd.DataFrame(
    {
        "feature": features_estandarizadas.columns,
        "media_estandarizada": features_estandarizadas.mean().values,
        "varianza_estandarizada": features_estandarizadas.var().values,
        "desviacion_estandarizada": features_estandarizadas.std().values,
    }
)

comprobacion_estandarizacion.to_csv(
    STANDARDIZATION_DIR / "comprobacion_estandarizacion.csv",
    index=False,
)
save_dataframe_image(
    comprobacion_estandarizacion,
    STANDARDIZATION_DIR / "comprobacion_estandarizacion.png",
    "Comprobacion de estandarizacion",
    figsize=(12, 4),
)
comprobacion_estandarizacion


## 18. Matriz R de correlacion de Pearson

Ahora calculamos la matriz `R = [rho_ij]` sobre las features estandarizadas. Cada entrada mide la relacion lineal entre dos variables y se interpreta entre -1 y 1: valores cercanos a 1 indican relacion positiva fuerte, valores cercanos a -1 indican relacion negativa fuerte, y valores cercanos a 0 indican poca relacion lineal.


In [ ]:
matriz_R = features_estandarizadas.corr(method="pearson")
matriz_R.to_csv(STANDARDIZATION_DIR / "matriz_correlacion_pearson_estandarizada.csv")
matriz_R


## 19. Heatmap de la matriz R

Ahora visualizamos la matriz `R` como mapa de calor. Esta vista permite detectar rapidamente pares de variables con alta correlacion que podrian ser redundantes o candidatas a fusionarse en experimentos posteriores.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8.5))
image = ax.imshow(matriz_R, cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Correlacion")

ax.set_xticks(range(len(matriz_R.columns)))
ax.set_yticks(range(len(matriz_R.index)))
ax.set_xticklabels(matriz_R.columns, rotation=45, ha="right")
ax.set_yticklabels(matriz_R.index)

for row in range(len(matriz_R.index)):
    for col in range(len(matriz_R.columns)):
        value = matriz_R.iloc[row, col]
        color = "white" if abs(value) >= 0.55 else "black"
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", color=color, fontsize=8)

ax.set_title("Matriz R de correlacion Pearson sobre features estandarizadas")
fig.tight_layout()
fig.savefig(
    STANDARDIZATION_DIR / "heatmap_correlacion_pearson_estandarizada.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()


## 20. Matriz de dispersion de features estandarizadas

Ahora generamos una matriz de dispersion para observar todos los pares de features. Coloreamos los puntos segun `condition` para buscar agrupamientos visuales, separaciones entre clases o relaciones no lineales que la correlacion de Pearson no siempre captura.


In [ ]:
columns = features_estandarizadas.columns.tolist()
count = len(columns)
target_values = sorted(df[target].dropna().unique())
palette = {
    target_values[0]: "#4C78A8",
    target_values[-1]: "#E45756",
} if len(target_values) >= 2 else {target_values[0]: "#4C78A8"}

fig, axes = plt.subplots(count, count, figsize=(count * 1.55, count * 1.55))

for row, y_column in enumerate(columns):
    for col, x_column in enumerate(columns):
        ax = axes[row, col]
        if row == col:
            ax.hist(features_estandarizadas[x_column], bins=18, color="#72B7B2", edgecolor="white")
        else:
            for target_value in target_values:
                mask = df[target] == target_value
                ax.scatter(
                    features_estandarizadas.loc[mask, x_column],
                    features_estandarizadas.loc[mask, y_column],
                    s=8,
                    alpha=0.55,
                    color=palette[target_value],
                    label=f"condition={target_value}",
                )

        if row == count - 1:
            ax.set_xlabel(x_column, fontsize=7, rotation=45)
        else:
            ax.set_xticklabels([])

        if col == 0:
            ax.set_ylabel(y_column, fontsize=7)
        else:
            ax.set_yticklabels([])

        ax.tick_params(axis="both", labelsize=6)
        ax.grid(alpha=0.15)

handles, labels = axes[0, 1].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper right", fontsize=9)

fig.suptitle("Matriz de dispersion de features estandarizadas", fontsize=16)
fig.tight_layout(rect=(0, 0, 0.98, 0.98))
fig.savefig(
    STANDARDIZATION_DIR / "matriz_dispersion_estandarizada.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()


## 21. PCA sobre features estandarizadas

Ahora aplicamos PCA para reducir las 13 dimensiones originales a componentes principales. PCA busca nuevas direcciones que concentran la mayor varianza posible; al proyectar los datos en 2D o 3D podemos observar si existe una estructura latente y si las clases de `condition` se agrupan de forma natural.


In [ ]:
import numpy as np

PCA_DIR = PROJECT_ROOT / "outputs" / "pca"
PCA_DIR.mkdir(parents=True, exist_ok=True)

x_pca = features_estandarizadas.to_numpy()
_, singular_values, vt = np.linalg.svd(x_pca, full_matrices=False)
pca_components = vt
pca_scores_array = x_pca @ pca_components.T
pca_eigenvalues = (singular_values ** 2) / (len(features_estandarizadas) - 1)
pca_explained_ratio = pca_eigenvalues / pca_eigenvalues.sum()
pca_cumulative_ratio = np.cumsum(pca_explained_ratio)

pca_component_names = [f"PC{i}" for i in range(1, len(pca_components) + 1)]
pca_scores = pd.DataFrame(
    pca_scores_array,
    columns=pca_component_names,
    index=features_estandarizadas.index,
)
pca_scores_with_target = pca_scores.copy()
pca_scores_with_target[target] = df[target]

pca_varianza = pd.DataFrame(
    {
        "componente": pca_component_names,
        "autovalor": pca_eigenvalues,
        "varianza_explicada": pca_explained_ratio,
        "varianza_acumulada": pca_cumulative_ratio,
    }
)
pca_cargas = pd.DataFrame(
    pca_components.T,
    index=features_estandarizadas.columns,
    columns=pca_component_names,
)

pca_scores_with_target.to_csv(PCA_DIR / "pca_scores.csv", index=False)
pca_varianza.to_csv(PCA_DIR / "pca_varianza_explicada.csv", index=False)
pca_cargas.to_csv(PCA_DIR / "pca_cargas_componentes.csv")

save_dataframe_image(
    pca_varianza,
    PCA_DIR / "pca_varianza_explicada.png",
    "Varianza explicada por PCA",
    figsize=(12, 4),
)
pca_varianza.head(8)


## 22. Varianza explicada por componentes

Ahora visualizamos cuanta informacion conserva cada componente principal. La barra muestra la varianza individual de cada componente y la linea muestra la varianza acumulada; esto permite decidir si 2D o 3D capturan suficiente estructura para una lectura visual razonable.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(1, len(pca_varianza) + 1)
ax.bar(x, pca_varianza["varianza_explicada"] * 100, color="#4C78A8", label="Individual")
ax.plot(
    x,
    pca_varianza["varianza_acumulada"] * 100,
    color="#E45756",
    marker="o",
    label="Acumulada",
)
ax.set_xlabel("Componente principal")
ax.set_ylabel("Varianza explicada (%)")
ax.set_title("Varianza explicada por PCA")
ax.set_xticks(x)
ax.set_ylim(0, 105)
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(PCA_DIR / "pca_scree_plot.png", dpi=180, bbox_inches="tight")
plt.show()


## 23. Proyeccion PCA en 2D

Ahora proyectamos los pacientes sobre los dos primeros componentes principales y coloreamos por `condition`. Si las clases se agrupan naturalmente, deberiamos ver regiones del plano donde predomine una clase sobre la otra.


In [ ]:
target_values = sorted(df[target].dropna().unique())
palette = {
    target_values[0]: "#4C78A8",
    target_values[-1]: "#E45756",
} if len(target_values) >= 2 else {target_values[0]: "#4C78A8"}

fig, ax = plt.subplots(figsize=(8, 6))
for target_value, color in palette.items():
    mask = df[target] == target_value
    ax.scatter(
        pca_scores.loc[mask, "PC1"],
        pca_scores.loc[mask, "PC2"],
        s=38,
        alpha=0.75,
        color=color,
        label=f"condition={target_value}",
        edgecolor="white",
        linewidth=0.4,
    )

pc1 = pca_varianza.loc[0, "varianza_explicada"] * 100
pc2 = pca_varianza.loc[1, "varianza_explicada"] * 100
ax.set_xlabel(f"PC1 ({pc1:.1f}% varianza)")
ax.set_ylabel(f"PC2 ({pc2:.1f}% varianza)")
ax.set_title("Proyeccion PCA 2D")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(PCA_DIR / "pca_2d.png", dpi=180, bbox_inches="tight")
plt.show()


## 24. Proyeccion PCA en 3D

Ahora repetimos la visualizacion usando los tres primeros componentes principales. Esta vista puede revelar separaciones que no son evidentes en 2D, especialmente cuando la tercera direccion conserva informacion util sobre la estructura de los datos.


In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

for target_value, color in palette.items():
    mask = df[target] == target_value
    ax.scatter(
        pca_scores.loc[mask, "PC1"],
        pca_scores.loc[mask, "PC2"],
        pca_scores.loc[mask, "PC3"],
        s=32,
        alpha=0.75,
        color=color,
        label=f"condition={target_value}",
    )

pc3 = pca_varianza.loc[2, "varianza_explicada"] * 100
ax.set_xlabel(f"PC1 ({pc1:.1f}%)")
ax.set_ylabel(f"PC2 ({pc2:.1f}%)")
ax.set_zlabel(f"PC3 ({pc3:.1f}%)")
ax.set_title("Proyeccion PCA 3D")
ax.legend()
fig.tight_layout()
fig.savefig(PCA_DIR / "pca_3d.png", dpi=180, bbox_inches="tight")
plt.show()


## 25. Funcion para graficar frecuencias

Ahora creamos una funcion reutilizable para visualizar cada feature. Si una columna numerica tiene muchos valores distintos usamos histograma; si tiene pocos valores distintos usamos barras, porque eso permite leer mejor variables discretas o categoricas codificadas.


In [ ]:
def plot_feature_frequency(series, feature_name, max_categories=20, bins=15, ax=None):
    current_ax = ax or plt.subplots(figsize=(8, 4.5))[1]
    clean_series = series.dropna()
    unique_count = clean_series.nunique()

    if pd.api.types.is_numeric_dtype(clean_series) and unique_count > max_categories:
        current_ax.hist(clean_series, bins=bins, edgecolor="black", color="#4C78A8")
        current_ax.set_xlabel(feature_name)
        current_ax.set_ylabel("Frecuencia")
    else:
        counts = clean_series.value_counts().sort_index()
        counts.plot(kind="bar", ax=current_ax, color="#59A14F", edgecolor="black")
        current_ax.set_xlabel(feature_name)
        current_ax.set_ylabel("Frecuencia")
        current_ax.tick_params(axis="x", rotation=45)

    current_ax.set_title(f"Frecuencia de {feature_name}")
    current_ax.grid(axis="y", alpha=0.25)
    return current_ax


## 26. Graficas individuales

Ahora calculamos las frecuencias feature por feature. Estas graficas ayudan a detectar distribuciones desequilibradas, valores raros, variables binarias y posibles columnas que necesiten transformacion antes de entrenar modelos.


In [ ]:
for feature in features:
    plot_feature_frequency(df[feature], feature)
    plt.tight_layout()
    plt.show()


## 27. Grafica combinada

Ahora juntamos todas las frecuencias en una sola figura. Esta vista sirve para comparar rapidamente las distribuciones y decidir que features necesitan una revision mas detallada.


In [ ]:
cols = 3
rows = math.ceil(len(features) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5.2, rows * 3.8))
flat_axes = axes.ravel() if hasattr(axes, "ravel") else [axes]

for ax, feature in zip(flat_axes, features):
    plot_feature_frequency(df[feature], feature, ax=ax)

for ax in flat_axes[len(features):]:
    ax.set_visible(False)

fig.suptitle("Graficas de frecuencia por feature", fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.98))
plt.show()


## 28. Guardado de resultados

Ahora guardamos las graficas en `outputs/graficas_frecuencia` para que el analisis sea reproducible y pueda entregarse junto al proyecto sin depender solo de la visualizacion del notebook.


In [ ]:
def clean_filename(name):
    valid = [char if char.isalnum() or char in ("-", "_") else "_" for char in name]
    return "".join(valid).strip("_") or "feature"

for feature in features:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_feature_frequency(df[feature], feature, ax=ax)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{clean_filename(feature)}_frecuencia.png", dpi=160)
    plt.close(fig)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5.2, rows * 3.8))
flat_axes = axes.ravel() if hasattr(axes, "ravel") else [axes]

for ax, feature in zip(flat_axes, features):
    plot_feature_frequency(df[feature], feature, ax=ax)

for ax in flat_axes[len(features):]:
    ax.set_visible(False)

fig.suptitle("Graficas de frecuencia por feature", fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.98))
fig.savefig(OUTPUT_DIR / "todas_las_frecuencias.png", dpi=180)
plt.close(fig)

print(f"Graficas guardadas en: {OUTPUT_DIR.resolve()}")
